# 12장. LLM이 만든 분석 코드를 검증하는 방법

이 노트북은 생성 코드를 **검토되지 않은 초안**으로 다룹니다. 위험 예제 코드는 문자열로만 정적 검사하며 실행하지 않습니다.

핵심 흐름: processed 입력 → 구조/키 → completed 집계 → 총합 → 정적 스캔 → 문제별 ML 누수 → 실행 Gate → 사람 승인 Evidence.


## 1. 프로젝트 경로와 processed 입력 확인


In [ ]:
from pathlib import Path
import sys

def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)
print('processed:', PROCESSED_DIR)
print('reports:', REPORT_DIR)


In [ ]:
from src.llm_code_validation import (
    DEFAULT_STATIC_SCAN_EXAMPLE,
    assert_validation_ready,
    build_code_review_checklist,
    build_dataset_inventory,
    build_error_fix_prompt_template,
    build_feature_audit,
    build_leakage_review_table,
    load_validation_data,
    run_llm_code_validation,
    safe_category_sales,
    safe_monthly_sales,
    scan_generated_code,
    validate_feature_list,
    validate_primary_keys,
    validate_relationship_keys,
    validate_required_columns,
)


## 2. 구조·PK·관계 검증

`order_items.order_item_id`도 PK 검증 대상이므로 필수 컬럼으로 확인합니다. 하나라도 FAIL이면 집계 전에 중단합니다.


In [ ]:
datasets = load_validation_data(PROCESSED_DIR)
inventory = build_dataset_inventory(datasets)
required_column_check = validate_required_columns(datasets)
primary_key_check = validate_primary_keys(datasets)
relationship_check = validate_relationship_keys(datasets)

display(inventory)
display(required_column_check)
display(primary_key_check)
display(relationship_check)

assert_validation_ready(required_column_check, primary_key_check, relationship_check)
print('구조·PK·관계 PASS')


## 3. completed 주문 집계와 총합 대조

`line_total = quantity × unit_price`를 검증하고 completed 주문만 집계합니다. source total과 category/monthly total 차이가 허용오차를 넘으면 즉시 중단합니다.


In [ ]:
category_sales, category_validation = safe_category_sales(
    datasets['order_items'], datasets['products'], datasets['orders']
)
monthly_sales, monthly_validation = safe_monthly_sales(
    datasets['order_items'], datasets['orders']
)
display(category_sales)
display(category_validation)
display(monthly_sales)
display(monthly_validation)


## 4. 위험 코드 AST 정적 스캔 — 실행 금지

아래 문자열에는 네트워크 요청과 파일 쓰기가 의도적으로 포함되어 있습니다. **문자열을 실행하지 않고 AST만 파싱**합니다. 탐지 0건도 안전 보증이 아닙니다.


In [ ]:
print(DEFAULT_STATIC_SCAN_EXAMPLE)
static_scan = scan_generated_code(DEFAULT_STATIC_SCAN_EXAMPLE)
display(static_scan)


## 5. 회귀와 분류의 Feature Contract를 분리해서 검토

회귀와 분류는 같은 금지 목록을 쓰지 않습니다. Chapter 09 회귀에서는 target 계산 재료가 누수이고, Chapter 10 분류에서는 주문 생성 시 이미 확정된 집계 특징을 교육용 가정 아래 사용할 수 있습니다.


In [ ]:
leakage_contract = build_leakage_review_table()
display(leakage_contract)

regression_features = ['payment_method', 'order_month', 'order_dayofweek', 'gender', 'age', 'city']
validate_feature_list(regression_features, problem='regression')
display(build_feature_audit(regression_features, problem='regression'))

classification_features = ['payment_method', 'item_count', 'total_quantity', 'order_amount', 'age', 'city']
validate_feature_list(classification_features, problem='classification')
display(build_feature_audit(classification_features, problem='classification'))


## 6. 오류 수정 Prompt와 사람 검토 Checklist


In [ ]:
display(build_code_review_checklist())
print(build_error_fix_prompt_template())


## 7. 전체 Evidence와 실행 Gate 생성

기본 정적 스캔 예시는 의도적으로 위험하므로 `execution_gate`가 BLOCKED/DO_NOT_EXECUTE를 보여 주는 것이 정상입니다. 자동 PASS가 실제 실행 승인을 의미하지 않습니다.


In [ ]:
result = run_llm_code_validation(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
)
display(result['outputs']['execution_gate'])
display(result['outputs']['sandbox_checklist'])
display(result['outputs']['package_review'])
display(result['outputs']['human_revision_log'])
for name, path in result['output_paths'].items():
    print(name, path)


## 정리

코드 실행 성공과 분석 타당성은 다릅니다. 정적 스캔도 안전 보증이 아닙니다. 구조·키·병합·completed 총합·문제별 누수·실행 위험을 검토하고, 승인된 코드만 credential 없는 제한 환경에서 실행한 뒤 결과를 다시 검증합니다.
